# IEEE-CIS Fraud Detection: Phase 1 EDA

This notebook inspects the joined training data before any imputation or feature engineering.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.clean import missingness_report, reduce_memory_usage
from src.data.load import load_training_data
from src.utils.config_loader import load_config

In [ ]:
data = load_training_data()
data = reduce_memory_usage(data)
data.shape

In [ ]:
class_counts = data['isFraud'].value_counts().sort_index()
class_percentages = data['isFraud'].value_counts(normalize=True).sort_index().mul(100)
class_summary = pd.DataFrame({'count': class_counts, 'percentage': class_percentages})
class_summary.index = class_summary.index.map({0: 'non-fraud', 1: 'fraud'})
print(class_summary.to_string(float_format=lambda value: f'{value:.2f}%'))

In [ ]:
missingness = missingness_report(data)
top_missing = missingness.head(20).sort_values('missing_pct')
ax = top_missing['missing_pct'].plot.barh(figsize=(10, 7), color='tab:purple')
ax.set(title='Top 20 columns by missingness', xlabel='Missing values (%)', ylabel='Column')
plt.tight_layout()

In [ ]:
amounts = data[['TransactionAmt', 'isFraud']].dropna()
positive_amounts = amounts[amounts['TransactionAmt'] > 0]
bins = np.logspace(np.log10(positive_amounts['TransactionAmt'].min()), np.log10(positive_amounts['TransactionAmt'].max()), 60)
fig, ax = plt.subplots(figsize=(10, 5))
for fraud_flag, label, color in [(0, 'non-fraud', 'tab:blue'), (1, 'fraud', 'tab:red')]:
    values = positive_amounts.loc[positive_amounts['isFraud'] == fraud_flag, 'TransactionAmt']
    ax.hist(values, bins=bins, density=True, alpha=0.55, label=label, color=color)
ax.set_xscale('log')
ax.set(title='Transaction amount distribution by class', xlabel='Transaction amount (log scale)', ylabel='Density')
ax.legend()
plt.tight_layout()

In [ ]:
config = load_config()
ordered_time = data['TransactionDT'].sort_values().reset_index(drop=True)
train_end = int(len(ordered_time) * config['temporal_split']['train_ratio'])
val_end = train_end + int(len(ordered_time) * config['temporal_split']['val_ratio'])
time_bins = pd.cut(data['TransactionDT'], bins=100)
counts = data.groupby(time_bins, observed=True).size()
centers = np.array([interval.mid for interval in counts.index])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(centers, counts.values, linewidth=1)
ax.axvline(ordered_time.iloc[train_end - 1], color='tab:orange', linestyle='--', label='train / val boundary')
ax.axvline(ordered_time.iloc[val_end - 1], color='tab:red', linestyle='--', label='val / test boundary')
ax.set(title='Transaction volume over relative time', xlabel='TransactionDT (seconds)', ylabel='Transactions per time bin')
ax.legend()
plt.tight_layout()

In [ ]:
categorical_columns = ['card4', 'card6', 'P_emaildomain', 'DeviceType']
cardinality = data[categorical_columns].nunique(dropna=False).sort_values(ascending=False)
print('Cardinality (including missing values):')
print(cardinality.to_string())